# **|| AI DevOps Log Analyzer — FastAPI Deployment ||**

This notebook turns the Log Analyzer into a running API server using FastAPI.

Instead of running cells manually, the server runs in the background and accepts requests automatically.

**Endpoints:**
- `POST /analyze` — send log text, get back anomalies and incident report
- `GET /health` — check if the server is running

**Tech used:** `FastAPI`, `Groq`, `ChromaDB`, `Sentence Transformers`, `localtunnel`


## **1** - ***Installation***

In [ ]:
%%capture
!pip install fastapi uvicorn groq chromadb sentence-transformers nest-asyncio
!npm install -g localtunnel

## **2** - ***API Key Setup***

We use environment variables to store the key safely. It never appears in the code.

> Free Groq API key at https://console.groq.com

In [ ]:
import os
from getpass import getpass

os.environ["GROQ_API_KEY"] = getpass("Enter your Groq API key: ")
print("Key saved!")

## **3** - ***Imports***

In [ ]:
import os
import re
import time
import threading
import subprocess
import requests
import nest_asyncio
import uvicorn

from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel

from groq import Groq
import chromadb
from sentence_transformers import SentenceTransformer

os.environ["PYTHONIOENCODING"] = "utf-8"
nest_asyncio.apply()

print("Imports done!")

## **4** - ***Settings***

In [ ]:
# --- Settings ---
MODEL_NAME     = "llama-3.1-8b-instant"
EMBED_MODEL    = "all-MiniLM-L6-v2"
CHUNK_SIZE     = 10
TOP_K          = 5
MAX_TOKENS     = 1024
TEMPERATURE    = 0.3
ANOMALY_LEVELS = ["ERROR", "CRITICAL", "FATAL", "WARNING", "WARN", "EXCEPTION", "TRACEBACK"]

client   = Groq(api_key=os.environ["GROQ_API_KEY"])
embedder = SentenceTransformer(EMBED_MODEL)

print("Settings loaded!")
print("Embedding model loading... (first time takes around 30 seconds)")

## **5** - ***Core Logic***

Parse logs, detect anomalies, build RAG index, analyze root causes and generate report.

In [ ]:
def parse_log_line(line):
    """Parses a log line into timestamp, level and message"""
    pattern = r"(\d{4}-\d{2}-\d{2}\s\d{2}:\d{2}:\d{2})\s+(\w+)\s+(.*)"
    match = re.match(pattern, line.strip())
    if match:
        return {
            "timestamp": match.group(1),
            "level":     match.group(2).upper(),
            "message":   match.group(3),
            "raw":       line.strip()
        }
    else:
        return {
            "timestamp": "unknown",
            "level":     "UNKNOWN",
            "message":   line.strip(),
            "raw":       line.strip()
        }


def detect_anomalies(parsed_logs):
    """Flags log entries that match anomaly levels"""
    return [log for log in parsed_logs if log["level"] in ANOMALY_LEVELS]


def build_rag_index(parsed_logs):
    """Chunks logs, embeds them and stores in ChromaDB"""
    chroma_client = chromadb.Client()
    try:
        chroma_client.delete_collection("logs")
    except:
        pass

    collection = chroma_client.create_collection("logs")
    chunks = []
    for i in range(0, len(parsed_logs), CHUNK_SIZE):
        batch = parsed_logs[i:i + CHUNK_SIZE]
        chunks.append({
            "id":   f"chunk_{i}",
            "text": "\n".join([log["raw"] for log in batch])
        })

    texts      = [c["text"] for c in chunks]
    ids        = [c["id"]   for c in chunks]
    embeddings = embedder.encode(texts).tolist()
    collection.add(documents=texts, embeddings=embeddings, ids=ids)
    return collection


def retrieve_chunks(collection, query):
    """Retrieves the most relevant log chunks for a query"""
    query_embedding = embedder.encode([query]).tolist()
    results = collection.query(query_embeddings=query_embedding, n_results=TOP_K)
    return results["documents"][0]


def call_llm(system_prompt, user_prompt):
    """Calls Groq LLM with basic retry"""
    try:
        response = client.chat.completions.create(
            model=MODEL_NAME,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user",   "content": user_prompt}
            ],
            max_tokens=MAX_TOKENS,
            temperature=TEMPERATURE
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        print(f"LLM error: {e}. Retrying in 5 seconds...")
        time.sleep(5)
        response = client.chat.completions.create(
            model=MODEL_NAME,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user",   "content": user_prompt}
            ],
            max_tokens=MAX_TOKENS,
            temperature=TEMPERATURE
        )
        return response.choices[0].message.content.strip()


def analyze_root_causes(anomalies, collection):
    """Uses RAG + LLM to suggest root cause for each unique anomaly"""
    system_prompt = """
        You are a senior DevOps engineer analyzing server logs.
        Given a log anomaly and relevant surrounding context, suggest the most likely root cause in 2-3 sentences.
        Be specific and technical. Only use the log context provided.
    """
    seen = set()
    results = []

    for anomaly in anomalies:
        key = anomaly["message"][:50]
        if key in seen:
            continue
        seen.add(key)

        relevant_chunks = retrieve_chunks(collection, f"{anomaly['level']} {anomaly['message']}")
        context = "\n".join(relevant_chunks)

        user_prompt = f"""Anomaly:
Timestamp: {anomaly['timestamp']}
Level: {anomaly['level']}
Message: {anomaly['message']}

Relevant log context:
{context}

What is the most likely root cause?"""

        root_cause = call_llm(system_prompt, user_prompt)
        results.append({
            "timestamp":  anomaly["timestamp"],
            "level":      anomaly["level"],
            "message":    anomaly["message"],
            "root_cause": root_cause
        })
        time.sleep(0.5)

    return results


def generate_report(anomalies, root_cause_results, parsed_logs):
    """Generates the final structured incident report"""
    system_prompt = """
        You are a DevOps engineer writing a formal incident report.
        Based on the anomalies and root causes provided, write a structured report in Markdown.

        Sections required:
        1. ## Incident Summary
        2. ## Timeline of Events
        3. ## Detected Anomalies
        4. ## Root Cause Analysis
        5. ## Recommended Actions
        6. ## Severity Assessment

        Be concise and technical. Only use information provided.
    """
    anomaly_summary = ""
    for item in root_cause_results:
        anomaly_summary += f"- [{item['level']}] {item['timestamp']}: {item['message']}\n"
        anomaly_summary += f"  Root cause: {item['root_cause']}\n\n"

    user_prompt = f"""Log analysis:
Total entries: {len(parsed_logs)}
Total anomalies: {len(anomalies)}
Time range: {parsed_logs[0]['timestamp']} to {parsed_logs[-1]['timestamp']}

Anomalies and root causes:
{anomaly_summary}"""

    return call_llm(system_prompt, user_prompt)


print("Core logic ready!")

## **6** - ***FastAPI App***

Defines the API endpoints. Accepts log text and returns the full analysis as JSON.

In [ ]:
app = FastAPI(
    title="AI DevOps Log Analyzer",
    description="Analyzes server logs, detects anomalies and generates incident reports using RAG",
    version="1.0.0"
)

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"]
)


class LogRequest(BaseModel):
    logs: str


@app.get("/health")
def health_check():
    return {"status": "running", "message": "Log Analyzer API is up"}


@app.post("/analyze")
def analyze_logs(request: LogRequest):
    """
    Accepts raw log text and returns:
    - list of detected anomalies
    - root cause for each anomaly
    - full incident report in markdown
    """
    if not request.logs.strip():
        raise HTTPException(status_code=400, detail="No log content provided")

    print("Received log analysis request...")

    parsed_logs = []
    for line in request.logs.splitlines():
        if line.strip():
            parsed_logs.append(parse_log_line(line))

    if not parsed_logs:
        raise HTTPException(status_code=400, detail="Could not parse any log entries")

    print(f"Parsed {len(parsed_logs)} log entries")

    anomalies = detect_anomalies(parsed_logs)
    print(f"Found {len(anomalies)} anomalies")

    if not anomalies:
        return {
            "total_entries":   len(parsed_logs),
            "anomalies_found": 0,
            "anomalies":       [],
            "root_causes":     [],
            "incident_report": "No anomalies detected in the provided logs."
        }

    collection = build_rag_index(parsed_logs)
    print("RAG index built")

    root_cause_results = analyze_root_causes(anomalies, collection)
    print("Root cause analysis done")

    report = generate_report(anomalies, root_cause_results, parsed_logs)
    print("Incident report generated")

    return {
        "total_entries":   len(parsed_logs),
        "anomalies_found": len(anomalies),
        "anomalies": [
            {"timestamp": a["timestamp"], "level": a["level"], "message": a["message"]}
            for a in anomalies
        ],
        "root_causes":     root_cause_results,
        "incident_report": report
    }


print("FastAPI app defined!")

## **7** - ***Start the Server***

Starts the FastAPI server in a background thread and exposes it publicly using localtunnel.
No account needed.

In [ ]:
# Start FastAPI in a background thread so the notebook doesn't get blocked
def run_server():
    config = uvicorn.Config(app, host="0.0.0.0", port=8000, log_level="warning")
    server = uvicorn.Server(config)
    server.run()

thread = threading.Thread(target=run_server)
thread.daemon = True
thread.start()

# Wait until the server is actually responding before continuing
max_wait = 30
for i in range(max_wait):
    try:
        r = requests.get("http://localhost:8000/health")
        if r.status_code == 200:
            print(f"Server is up after {i+1} seconds!")
            break
    except:
        time.sleep(1)
else:
    print("Server did not start in time")

# Start localtunnel to expose the server publicly
tunnel = subprocess.Popen(
    ["lt", "--port", "8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

# Read the public URL from localtunnel output
time.sleep(2)
output = tunnel.stdout.readline().decode().strip()
BASE_URL = output.replace("your url is: ", "").strip()

print(f"\nServer is live at: {BASE_URL}")
print(f"API docs: {BASE_URL}/docs")
print(f"Analyze endpoint: {BASE_URL}/analyze")
print("\nReady — run the test cell below!")

## **8** - ***Test the API***

Run this cell right after the server cell. Uses the same BASE_URL automatically.

In [ ]:
from rich.console import Console
from rich.panel import Panel
from rich.table import Table
from rich.markdown import Markdown

console = Console()

sample_logs = """
2024-01-15 08:00:01 INFO Server started successfully on port 8080
2024-01-15 08:12:45 WARNING High memory usage detected: 87% of available RAM in use
2024-01-15 08:15:22 ERROR Database connection timeout after 30s
2024-01-15 08:16:25 CRITICAL Database connection failed — all retry attempts exhausted
2024-01-15 08:22:05 CRITICAL Out of memory — killing process 4821
2024-01-15 08:28:00 ERROR Backup failed — insufficient disk space
"""

console.print(Panel("[bold cyan]Sending request to API...[/bold cyan]", expand=False))
response = requests.post(
    f"{BASE_URL}/analyze",
    json={"logs": sample_logs},
    timeout=300 # 5 minutes - Enough time for the full pipeline
)

if response.status_code == 200:
    api_data = response.json()

    # show summary
    console.print(Panel(
        f"[bold green]Analysis Complete[/bold green]\n"
        f"Total log entries: {api_data['total_entries']}\n"
        f"Anomalies found: {api_data['anomalies_found']}",
        expand=False
    ))

    # show anomalies in a table
    table = Table(title="Detected Anomalies", show_lines=True)
    table.add_column("Timestamp", style="dim")
    table.add_column("Level", style="bold red")
    table.add_column("Message")
    for a in api_data["anomalies"]:
        color = "red" if a["level"] in ["ERROR", "CRITICAL"] else "yellow"
        table.add_row(a["timestamp"], f"[{color}]{a['level']}[/{color}]", a["message"])
    console.print(table)

    # show incident report
    console.print(Panel("[bright_cyan]Incident Report[/bright_cyan]", expand=False))
    console.print(Markdown(api_data["incident_report"]))
else:
    console.print(f"[red]Error: {response.status_code} — {response.text}[/red]")

## **9** - ***Upload Your Own Log File***

Instead of the sample logs above, you can upload your own `.log` or `.txt` file and send it to the API.

In [ ]:
from google.colab import files

print("Upload your log file (.log or .txt):")
uploaded = files.upload()

if uploaded:
    filename = list(uploaded.keys())[0]
    with open(filename, "r", errors="ignore") as f:
        log_content = f.read()

    console.print(Panel(f"[bold cyan]Loaded {len(log_content.splitlines())} lines from {filename}[/bold cyan]", expand=False))
    print("Sending to API...")

    response = requests.post(
        f"{BASE_URL}/analyze",
        json={"logs": log_content},
        timeout=300 # 5 minutes - Enough time for the full pipeline
    )

    if response.status_code == 200:
        api_data = response.json()

        # Show summary
        console.print(Panel(
            f"[bold green]Analysis Complete[/bold green]\n"
            f"Total log entries: {api_data['total_entries']}\n"
            f"Anomalies found: {api_data['anomalies_found']}",
            expand=False
        ))

        # Show anomalies in a table
        table = Table(title="Detected Anomalies", show_lines=True)
        table.add_column("Timestamp", style="dim")
        table.add_column("Level", style="bold red")
        table.add_column("Message")
        for a in api_data["anomalies"]:
            color = "red" if a["level"] in ["ERROR", "CRITICAL"] else "yellow"
            table.add_row(a["timestamp"], f"[{color}]{a['level']}[/{color}]", a["message"])
        console.print(table)

        # Show incident report
        console.print(Panel("[bright_cyan]Incident Report[/bright_cyan]", expand=False))
        console.print(Markdown(api_data["incident_report"]))
    else:
        console.print(f"[red]Error: {response.status_code} — {response.text}[/red]")
else:
    print("No file uploaded.")

## **10** - ***Save the Report***

Run this after either the test cell or the upload cell to save and download the results.

In [ ]:
import json as json_lib
from datetime import datetime

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

# save markdown report
md_filename = f"incident_report_{timestamp}.md"
with open(md_filename, "w") as f:
    f.write(f"# Incident Report\n")
    f.write(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n\n")
    f.write(api_data["incident_report"])

# save root cause analysis as json
json_filename = f"root_cause_analysis_{timestamp}.json"
with open(json_filename, "w") as f:
    json_lib.dump({
        "total_entries":   api_data["total_entries"],
        "anomalies_found": api_data["anomalies_found"],
        "anomalies":       api_data["anomalies"],
        "root_causes":     api_data["root_causes"]
    }, f, indent=2)

print(f"Saved: {md_filename}")
print(f"Saved: {json_filename}")

# auto download in Colab
try:
    from google.colab import files
    files.download(md_filename)
    files.download(json_filename)
    print("Download started!")
except ImportError:
    print("Files saved in current directory.")

## ****How to Run Locally (outside Colab)***

1. Install dependencies:
```
pip install fastapi uvicorn groq chromadb sentence-transformers
npm install -g localtunnel
```

2. Create a `.env` file:
```
GROQ_API_KEY=your_key_here
```

3. Make sure `.env` is in your `.gitignore`

4. Run the server:
```
uvicorn app:app --reload
```

5. In a separate terminal, start the tunnel:
```
lt --port 8000
```

6. Visit `http://localhost:8000/docs` for interactive API documentation